# EquiCave: train and evaluate the equivariant multi-task pocket model

Run this notebook **from inside a checkout of the repository** (its working directory must be the repository
root). The first cell installs the package in editable mode, after which `equicave` and `training` are ordinary
importable packages: no `sys.path` juggling and no shell-outs to scripts.

Stages: install and check the environment, build the data, train the network (GPU), evaluate every benchmark,
and draw the figures. Nothing here calls P2Rank, fpocket or Java.

In [ ]:
# Editable install of the package and the training extras (torch, transformers) plus matplotlib for the figures.
%pip -q install -e ".[train,viz,dev]"

import torch

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"torch {torch.__version__} | device {DEVICE}"
      + (f" | {torch.cuda.get_device_name(0)}, {torch.cuda.get_device_properties(0).total_memory / 2**30:.0f} GiB"
         if DEVICE.type == "cuda" else " | no GPU: the network stages below will be far too slow"))

In [ ]:
# The whole public API used in this notebook.
from pathlib import Path

import numpy as np
import pandas as pd

from equicave import detect, labels, metrics, peptide, pocket_features, pockets, structure, tables, viz
from training.common.config import load_config, pick_device
from training.pockets import data as netdata
from training.pockets import net_task

REPO = Path.cwd()
PROCESSED = REPO / "data/processed"
PDB_DIR = REPO / "data/pockets_ds/pdb"
RESULTS = REPO / "docs/results"
print("geometry constants used everywhere:", pocket_features.geometry())

## 1. Tests first

The suite is CPU-only and runs on synthetic structures. It checks, among other things, that the network is
exactly rotation-equivariant and that the chiral variant is *not* reflection-invariant. If it fails, stop here.

In [ ]:
import pytest

assert pytest.main(["-q", "tests"]) == 0, "test suite failed"

## 2. Data

`build_manifest` queries the RCSB PDB for X-ray protein structures with a drug-sized ligand, groups them by
30 %-identity cluster, assigns five folds per cluster and removes the held-out target families. Skip this cell if
`data/processed/manifest.csv` is already in the checkout.

In [ ]:
import subprocess, sys

def run(*args):
    """Run one of the repository scripts in this interpreter and show its output."""
    p = subprocess.run([sys.executable, *map(str, args)], text=True, capture_output=True)
    print(p.stdout[-4000:] or p.stderr[-4000:])
    assert p.returncode == 0, f"{args[0]} failed"

if not (PROCESSED / "manifest.csv").exists():
    run("scripts/data/build_manifest.py", "--n", "3200", "--seed", "0")
run("scripts/data/fetch_structures.py", "--jobs", "8")

manifest = pd.read_csv(PROCESSED / "manifest.csv")
print(f"{len(manifest)} entries, {manifest.cluster30.nunique()} clusters, folds {manifest.fold.value_counts().sort_index().tolist()}")
manifest.head(3)

## 3. What the candidate generator sees

One structure, end to end: free lattice points, 26-ray buriedness, connected cavities, peak splitting. The
figures below are the actual arrays the model is trained on, not an illustration.

In [ ]:
def first_with_ligand(pdb_dir, min_heavy=8, limit=50):
    """The first downloaded structure that has a drug-sized ligand, so the figures below show a real site."""
    for path in sorted(pdb_dir.glob("*.pdb"))[:limit]:
        ligands = structure.read_ligands(path, min_heavy=min_heavy)
        if ligands:
            return path, ligands
    raise FileNotFoundError("no structure with a ligand in the directory")

pdb, ligs = first_with_ligand(PDB_DIR)
st = structure.read_pdb(pdb)
cands = detect.detect_sites(st["xyz"])
lig_atoms = np.vstack([l["xyz"] for l in ligs])

print(f"{pdb.stem}: {len(st['xyz'])} heavy atoms, {len(cands)} candidates, {len(ligs)} ligand copies")
print(f"best DCA over candidates: {min(pockets.dca(c['center'], lig_atoms) for c in cands):.2f} Å (a hit is ≤ 4 Å)")

pd.DataFrame(detect.candidate_table(cands)).head(8)

In [ ]:
best_candidate = min(cands, key=lambda c: pockets.dca(c["center"], lig_atoms))

viz.plot_sites_overview(pdb, top_k=5)
viz.plot_pocket(pdb, best_candidate, ligands=ligs,
                title=f"{pdb.stem}: candidate rank {best_candidate['rank']} at the ligand "
                      f"(DCA {pockets.dca(best_candidate['center'], lig_atoms):.2f} Å)")
viz.plot_buriedness_slice(pdb)

## 4. Native candidates for the whole set, then the ranker

`build_native.py` writes one row per candidate with 32 features, the DCA/DCC distances and the label;
`train_ranker.py` cross-validates a LambdaRank model by cluster with several seeds and a paired cluster
bootstrap against the detector order.

In [ ]:
try:
    cand = tables.read_table(PROCESSED, "candidates_native")
except FileNotFoundError:
    run("scripts/train/build_native.py", "--jobs", "4")
    cand = tables.read_table(PROCESSED, "candidates_native")
run("scripts/train/train_ranker.py", "--seeds", "5", "--ablate", "--model", "models/ranker_native.txt")

print(f"{cand.pdb.nunique()} structures, {len(cand)} candidates, ceiling "
      f"{(cand.groupby('pdb').label.max() == 1).mean():.3f}, mean candidates {len(cand) / cand.pdb.nunique():.1f}")
viz.comparison_table()

In [ ]:
fig = viz.plot_metric_bars(viz.comparison_table(keys=("ranker_native",)), metric="top-1",
                           title="Ranking native candidates (5-fold cluster CV, 5 seeds)")

## 5. The network (GPU)

`net_task.run` featurises every structure once into a cache (residue nodes with ESM-2, cavity probes, surface
points), then trains the multi-task model on four folds and validates on the fifth. Each run writes
`model.pt`, `history.json`, `metrics.json` and `model_card.json`.

On a CPU use `training/configs/pockets_net_cpu.yaml`, whose numbers are a code-path check and nothing more.

In [ ]:
from types import SimpleNamespace

CFG = "training/configs/pockets_net.yaml" if DEVICE.type == "cuda" else "training/configs/pockets_net_cpu.yaml"
OUT = REPO / "runs/training/net"

args = SimpleNamespace(config=CFG, out=str(OUT), device=str(DEVICE),
                       set=["split.val_fold=0", "optim.seed=0", "tag=full"])
net_task.run(args)   # minutes per epoch on an A100; hours on a CPU

In [ ]:
# the trained model, its validation metrics and the loss curves
run_dir = sorted(OUT.glob("full_fold0_seed*"))[-1]
model, cfg = net_task.load_model(run_dir / "model.pt", DEVICE)
metrics_json = pd.read_json(run_dir / "metrics.json", typ="series")
history = pd.json_normalize(pd.read_json(run_dir / "history.json").to_dict("records"))
print(metrics_json[["res_ap", "occ_ap", "res_auroc", "occ_auroc", "occ_ece"]])
display(pd.DataFrame(metrics_json["net_sites"], index=["network alone"]))
ax = history.filter(like="train.").plot(figsize=(7, 3.5), title="training losses per epoch")
ax.set_xlabel("epoch"); ax.set_ylabel("loss")

### The hotspot field of a real pocket

The field is the network's per-class probability that a ligand atom of that class sits at a given cavity point.
The crystal ligand is drawn as triangles: it should fall where the matching classes light up.

In [ ]:
sample = netdata.load(sorted((REPO / cfg["data"]["cache_dir"]).glob("*.npz"))[0])
batch = netdata.to_torch(sample, DEVICE)
with torch.no_grad():
    out = model(batch)

probe_pos = sample["pos"][sample["n_res"]:sample["n_res"] + sample["n_probe"]]
predicted = torch.sigmoid(out["hot_logit"]).cpu().numpy()
near = np.linalg.norm(probe_pos - sample["site_centers"][0], axis=1) <= 12.0

occupancy_auroc = metrics.auroc(sample["y_occ"], torch.sigmoid(out["occ_logit"]).cpu().numpy())
print(f"{sample['pdb']}: {int(near.sum())} probes around the first site, "
      f"probe-occupancy AUROC {occupancy_auroc:.3f}")

viz.plot_hotspot_field(probe_pos[near], predicted[near], labels.HOTSPOT_CLASSES,
                       ligand=sample["lig_xyz"], threshold=0.3)

# the same panels built from the crystal labels, as the target the field is trying to reproduce
viz.plot_hotspot_field(probe_pos[near], sample["y_hot"][near], labels.HOTSPOT_CLASSES,
                       ligand=sample["lig_xyz"], threshold=0.5)

### Network features for the hybrid ranker

One model per fold, so a candidate is always scored by a network that never saw its structure; the features then
join the ranker.

In [ ]:
net_task.run(SimpleNamespace(config=CFG, out=str(REPO / "runs/training/net-oof"), device=str(DEVICE),
                            set=["mode=oof", "tag=full"]))
run("scripts/train/train_ranker.py", "--features-extra", "net", "--seeds", "5")
viz.comparison_table()

### Ablations

`full`, `no_probes`, `no_surface`, `no_esm`, `no_tensors` (degree ≤ 1), `no_vectors`, `invariant` (distances only,
same depth and width) and `achiral`, three seeds each. This is the evidence for the architecture claims.

In [ ]:
for ablation in ["full", "no_probes", "no_surface", "no_esm", "no_tensors", "invariant"]:
    for seed in range(3):
        net_task.run(SimpleNamespace(config=CFG, out=str(REPO / "runs/training/ablations"), device=str(DEVICE),
                                    set=[f"ablation={ablation}", f"optim.seed={seed}", "split.val_fold=0",
                                         f"tag={ablation}"]))

run("scripts/train/collect_ablations.py", "--runs", "runs/training/ablations", "--out", "docs/results/ablations.md")

## 6. Benchmarks

`fetch_eval_sets.py` downloads the published lists (PDB ids only, from their original sources) and
`evaluate.py` applies one protocol to all of them: own candidates, the ranker, the network, DCA and DCC,
top-1 / top-3 / top-N / top-(N+2), non-redundant predictions, and a separate row for structures that share a
30 %-identity cluster with training.

In [ ]:
run("scripts/data/fetch_eval_sets.py")

model_path = run_dir / "model.pt"
for benchmark in ["heldout", "coach420", "holo4k"]:
    run("scripts/eval/evaluate.py", "--set", benchmark, "--ranker", "models/ranker_native.txt",
        "--net", model_path, "--jobs", "4")

ours = viz.eval_table()
ours

### Published numbers of competing methods, next to ours

The published column is context, not a comparison: those papers use different ligand filters, chain handling and
splits. The honest comparison is `scripts/baselines/run_external.py`, which re-runs fpocket and P2Rank on our
structures and our labels (next cell).

In [ ]:
published = viz.published_reference()
print(published.attrs["note"])

def top_n(benchmark: str, method_contains: str = "ranker (DCA)"):
    """Our measured top-N on one benchmark, from docs/results."""
    t = ours[(ours.benchmark == benchmark) & ours.method.str.contains(method_contains, regex=False)]
    return t["top-N"].iloc[0] if len(t) else "not run"

summary = published.copy()
summary.loc[len(summary)] = ["EquiCave (ranker)", "our candidates + LambdaRank",
                             top_n("coach420"), top_n("holo4k"), "not run"]
summary.loc[len(summary)] = ["EquiCave (ranker + network)", "full model",
                             top_n("coach420", "ranker+net"), top_n("holo4k", "ranker+net"), "not run"]
summary

In [ ]:
# the paired comparison: fpocket and P2Rank on exactly our structures and labels (needs FPOCKET / PRANK and Java)
import os

if os.environ.get("FPOCKET") or os.environ.get("PRANK"):
    for tool in filter(None, ["fpocket" if os.environ.get("FPOCKET") else None,
                              "p2rank" if os.environ.get("PRANK") else None]):
        run("scripts/baselines/run_external.py", "--tool", tool, "--jobs", "4")
    run("scripts/eval/compare_methods.py", "--tags", "native:nat_rank", "fpocket:tool_rank", "p2rank:tool_rank")
    display(viz.comparison_table(keys=("methods_comparison",)))
    viz.plot_metric_bars(viz.comparison_table(keys=("methods_comparison",)), metric="top-1",
                        title="Same structures, same labels: candidate generators and rankers")
else:
    print("set FPOCKET and/or PRANK to run the external baselines; they are optional and never used by the model")

## 7. Peptide-binding sites

A peptide groove is shallow and elongated, so it needs its own candidate tier and its own features (anisotropy,
exposed receptor backbone). The benchmark is built from RCSB protein-peptide complexes whose receptor clusters are
disjoint from the small-molecule training clusters, and the peptide chains are removed from the input.

In [ ]:
if not (PROCESSED / "manifest_peptide.csv").exists():
    run("scripts/data/build_peptide_manifest.py", "--n", "4000", "--seed", "0",
        "--disjoint-from", PROCESSED / "manifest.csv")
    run("scripts/data/fetch_structures.py", "--manifest", PROCESSED / "manifest_peptide.csv", "--jobs", "8")
try:
    tables.find_table(PROCESSED, "candidates_peptide")
except FileNotFoundError:
    run("scripts/train/build_peptide.py", "--jobs", "4", "--min-res", "3")
run("scripts/train/train_ranker.py", "--tag", "peptide", "--seeds", "5", "--ablate",
    "--model", "models/ranker_peptide.txt")

viz.comparison_table(keys=("ranker_peptide",))

In [ ]:
# one peptide complex: the receptor without its peptide, the groove candidates, and the peptide itself
pep_manifest = pd.read_csv(PROCESSED / "manifest_peptide.csv")
row = next(r for _, r in pep_manifest.iterrows() if (PDB_DIR / f"{r.pdb}.pdb").exists())
pep_pdb = PDB_DIR / f"{row.pdb}.pdb"

peptides = structure.read_peptide_ligands(pep_pdb, min_res=3, max_res=30)
receptor = "".join(c for c in structure.receptor_chains(pep_pdb) if c not in {p["chain"] for p in peptides})
rec = structure.read_pdb(pep_pdb, receptor)

grooves = peptide.detect_peptide_sites(rec["xyz"], rec, min_points=20, min_anisotropy=1.5, nms=4.0)
cavities = detect.detect_sites(rec["xyz"], max_sites=30)
merged = peptide.merge_with_small_molecule(grooves, cavities)

pep_atoms = np.vstack([p["xyz"] for p in peptides])
print(f"{row.pdb}: peptide {[(p['comp'], p['n_res'], p['seq']) for p in peptides]}")
print(f"{len(cavities)} cavity + {len(grooves)} groove candidates; best DCA "
      f"{min(pockets.dca(c['center'], pep_atoms) for c in merged):.2f} Å")

pd.DataFrame(peptide.groove_features(merged, rec)).assign(
    dca=[pockets.dca(c["center"], pep_atoms) for c in merged]).sort_values("dca").head(6)

In [ ]:
hit = min(merged, key=lambda c: pockets.dca(c["center"], pep_atoms))
fig = viz.plot_pocket(pep_pdb, hit, ligands=[dict(comp="peptide", xyz=pep_atoms)],
                      title=f"{row.pdb}: groove candidate at the peptide (tier {hit['tier']}, "
                            f"length {hit.get('length', float('nan')):.0f} Å)")

## 8. What to report back

- `runs/training/*/model_card.json` — config, geometry constants, data scope and validation metrics of every run.
- `docs/results/*.md` and `*.json` — every table, each with its protocol in the header.
- `history.json` — the loss curves, to show that a run converged.

Rules: no number from a run that did not converge, three seeds or more for anything in a table, and "not run"
written wherever a number is missing.